# Tokenization

In [2]:
import tiktoken

text = "Студенты любят практику и хотят понять, как LLM видит текст."

encoding = tiktoken.encoding_for_model("gpt-5")
token_ids = encoding.encode(text)

print(token_ids)

# Вывод самих токенов
tokens = [encoding.decode([token_id]) for token_id in token_ids]
print(tokens)

print(len(token_ids))

[44098, 3388, 29814, 171190, 32399, 4374, 816, 141869, 102970, 11, 6220, 451, 19641, 11431, 1108, 60350, 13]
['Ст', 'уд', 'енты', ' любят', ' практи', 'ку', ' и', ' хотят', ' понять', ',', ' как', ' L', 'LM', ' вид', 'ит', ' текст', '.']
17


## Tokenization functions

In [21]:
import tiktoken

# Получение токенайзера
def get_openai_encoding(model_name: str):
    try:
        return tiktoken.encoding_for_model(model_name) # Пытается получить кодировщик, «зашитый» под конкретную модель
    except KeyError: # Если имя модели неизвестно tiktoken (маппинга нет), выбрасывается KeyError
        return tiktoken.get_encoding("o200k_base") # Возвращает кодировщик по явному имени

# Считает количество токенов 
def count_tokens(text: str, model_name: str) -> int:
    encoding = get_openai_encoding(model_name)
    token_ids = encoding.encode(text)
    return len(token_ids)

# Просматриваем сами токены
def show_tokens(text: str, model_name: str) -> list[str]:
    encoding = get_openai_encoding(model_name)
    token_ids = encoding.encode(text)
    return [encoding.decode([token_id]) for token_id in token_ids]

In [23]:
# Простой пример

s = 'A-134025, я не знаю, где тебя искать'
m = 'gpt-4'

print(f'Кодировщик для GPT-4: {get_openai_encoding(m)}') # получаем токенайзер для gpt-4
print(show_tokens(s, m))
print(count_tokens(s, m))

Кодировщик для GPT-4: <Encoding 'cl100k_base'>
['A', '-', '134', '025', ',', ' я', ' не', ' з', 'на', 'ю', ',', ' г', 'де', ' т', 'еб', 'я', ' и', 'ск', 'ать']
19


In [5]:
# Пример на разных строках

texts = {
    "english": "Tokenization helps estimate the context size before sending a request.",
    "russian": "Токенизация помогает заранее оценить размер контекста.",
    "mixed": "Пользователь asked the модель to summarize 3 JSON файла.",
    "code": "def calculate_price(price, discount): return price * (1 - discount)",
    "numbers": "order_id=984120, sku_id=100239991, date=2026-05-10",
    "emoji": "🛠 ℹ ⚖ 📐 🔑 🏧.",
}

for name, text in texts.items():
    print(
        name,
        "characters:",
        len(text),
        "tokens:",
        count_tokens(text, 'gpt-4'),
    ) 

english characters: 70 tokens: 12
russian characters: 54 tokens: 22
mixed characters: 56 tokens: 14
code characters: 67 tokens: 15
numbers characters: 50 tokens: 21
emoji characters: 12 tokens: 17


## Picking up tokenizer

In [24]:
from transformers import AutoTokenizer 

# скачиваем токенайзер для модели TinyLlama
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

In [63]:
# Применяем автоматически подобранный токенайзер
 
text = "Студенты любят практику и хотят понять, как LLM видит текст."

encoded_new = tokenizer(text, add_special_tokens=False)

token_ids_new = encoded_new["input_ids"]
tokens_new = tokenizer.convert_ids_to_tokens(token_ids_new)

print(token_ids_new)
print(tokens_new)
print(len(tokens_new))

[857, 1500, 4466, 3327, 6331, 29975, 10691, 4460, 9054, 1382, 606, 6785, 6566, 29932, 733, 1200, 1413, 29892, 5413, 365, 26369, 2196, 956, 29932, 23567, 464, 29889]
['▁С', 'ту', 'ден', 'ты', '▁лю', 'б', 'ят', '▁пра', 'кти', 'ку', '▁и', '▁хо', 'тя', 'т', '▁по', 'ня', 'ть', ',', '▁как', '▁L', 'LM', '▁ви', 'ди', 'т', '▁тек', 'ст', '.']
27


In [64]:
# Применяем токенайзер HuggingFace

def count_hf_tokens(text: str) -> int:
    encoded = tokenizer(text, add_special_tokens=False)
    return len(encoded["input_ids"])

def show_hf_tokens(text: str) -> list[str]:
    encoded = tokenizer(text, add_special_tokens=False)
    token_ids = encoded["input_ids"]
    return tokenizer.convert_ids_to_tokens(token_ids)

def undoing(text: str) -> int:
    token_ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    return tokenizer.decode(token_ids)

text = "Студенты любят практику❤️"

print(show_hf_tokens(text))
print(count_hf_tokens(text)) 
print(undoing(text))

['▁С', 'ту', 'ден', 'ты', '▁лю', 'б', 'ят', '▁пра', 'кти', 'ку', '<0xE2>', '<0x9D>', '<0xA4>', '️']
14
Студенты любят практику❤️


## Special tokens

In [9]:
'''
Специальные токены — это отдельные токены со своими id в словаре. Они добавляются для технических целей:  
'<s>' — начало последовательности;
'</s>' — конец последовательности;
'[CLS]' — начало входа в BERT-подобных моделях;
'[SEP]' — разделитель;
'<pad>' или '[PAD]' — выравнивание длины;
'<unk>' или '[UNK]' — неизвестный фрагмент.

Служебные обозначения внутри токенов — это способ показать, как токен связан с соседними частями текста:  
'▁' часто показывает пробел или начало слова;
'##' показывает продолжение предыдущего слова;  
'''

"\nСпециальные токены — это отдельные токены со своими id в словаре. Они добавляются для технических целей:  \n'<s>' — начало последовательности;\n'</s>' — конец последовательности;\n'[CLS]' — начало входа в BERT-подобных моделях;\n'[SEP]' — разделитель;\n'<pad>' или '[PAD]' — выравнивание длины;\n'<unk>' или '[UNK]' — неизвестный фрагмент.\n\nСлужебные обозначения внутри токенов — это способ показать, как токен связан с соседними частями текста:  \n'▁' часто показывает пробел или начало слова;\n'##' показывает продолжение предыдущего слова;  \n"

In [40]:
# Те же функции, но со специальными символами


def count_hf_tokens(text: str) -> int:
    encoded = tokenizer(text, add_special_tokens=True)
    return len(encoded["input_ids"])

def show_hf_tokens(text: str) -> list[str]:
    encoded = tokenizer(text, add_special_tokens=True)
    token_ids = encoded["input_ids"]
    return tokenizer.convert_ids_to_tokens(token_ids)


def undoing(text: str) -> int:
    token_ids = tokenizer(text, add_special_tokens=True)["input_ids"]
    return tokenizer.decode(token_ids)

print(show_hf_tokens(text))
print(count_hf_tokens(text)) 
print(undoing(text))

['<s>', '▁С', 'ту', 'ден', 'ты', '▁лю', 'б', 'ят', '▁пра', 'кти', 'ку', '<0xE2>', '<0x9D>', '<0xA4>', '️']
15
<s> Студенты любят практику❤️


In [41]:
# Маппинг специальных символов для HuggingFace токенайзера 

print(tokenizer.special_tokens_map)

{'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '</s>'}


## tiktoken vs AutoTokenizer

In [46]:
texts = {
    "english": "Tokenization helps estimate the context size before sending a request.",
    "russian": "Токенизация помогает заранее оценить размер контекста.",
    "mixed": "Пользователь asked the модель to summarize 3 JSON файла.",
    "code": "def calculate_price(price, discount): return price * (1 - discount)",
    "numbers": "order_id=984120, sku_id=100239991, date=2026-05-10",
    "emoji": "🛠 ℹ ⚖ 📐 🔑 🏧.",
} 

In [47]:
import pandas as pd

rows = []

for text_type, text in texts.items():
    gpt_tokens = count_tokens(text, 'gpt-4')
    llama_tokens = count_hf_tokens(text)

    rows.append(
        {
            "text_type": text_type,
            "characters": len(text),
            "gpt_tokens": gpt_tokens,
            "llama_tokens": llama_tokens,
            "difference": llama_tokens - gpt_tokens,
        }
    )

comparison_df = pd.DataFrame(rows)

print(comparison_df)

  text_type  characters  gpt_tokens  llama_tokens  difference
0   english          70          12            13           1
1   russian          54          22            22           0
2     mixed          56          14            18           4
3      code          67          15            20           5
4   numbers          50          21            39          18
5     emoji          12          17            30          13


## Requests' cost

In [13]:
# Условия 

gpt_input_tokens = 78
llama_input_tokens = 112
expected_output_tokens = 500

gpt_input_price = 0.75
gpt_output_price = 4.50

llama_input_price = 0.60
llama_output_price = 0.60

budget = 20
requests_count = 10000

In [14]:
def estimate_cost(
    input_tokens: int,
    output_tokens: int,
    input_price_per_million: float,
    output_price_per_million: float,
) -> float:
    input = input_tokens * input_price_per_million
    output = output_tokens * output_price_per_million
    return input + output


In [15]:
gpt_cost = estimate_cost(gpt_input_tokens, expected_output_tokens, gpt_input_price/1000000, gpt_output_price/1000000)
gpt_total_cost = gpt_cost * requests_count

llama_cost = estimate_cost(llama_input_tokens, expected_output_tokens, llama_input_price/1000000, llama_output_price/1000000)
llama_total_cost = llama_cost * requests_count

print("GPT total cost:", gpt_total_cost)
print("LLaMA total cost:", llama_total_cost)

GPT total cost: 23.085
LLaMA total cost: 3.6719999999999997


In [16]:

if gpt_total_cost <= 20 and llama_total_cost <= 20:
    print("Обе модели проходят по бюджету")
elif gpt_total_cost <= 20 and llama_total_cost > 20:
    print("По бюджету проходит GPT")
elif llama_total_cost <= 20 and gpt_total_cost > 20:
    print("По бюджету проходит LLaMA")
else:
    print("Ни одна модель не проходит по бюджету")

По бюджету проходит LLaMA


# Embedding

In [74]:
# Выводим токены и их id

texts = [
    "Старый замок стоял на холме.",
    "Сломался дверной замок."
]

for text in texts:
    tokens = tokenizer.tokenize(text) # разбиваем каждый элемент текста на токены
    token_ids = tokenizer.encode(text, add_special_tokens=False) # кладем идентификаторы токенов 

    print(text)
    print(tokens)
    print(token_ids)
    print()


Старый замок стоял на холме.
['▁Ста', 'ры', 'й', '▁за', 'мо', 'к', '▁', 'стоя', 'л', '▁на', '▁хо', 'л', 'ме', '.']
[6603, 3541, 29977, 1077, 1630, 29951, 29871, 6428, 29944, 665, 6785, 29944, 1488, 29889]

Сломался дверной замок.
['▁С', 'ло', 'ма', 'лся', '▁д', 'вер', 'ной', '▁за', 'мо', 'к', '.']
[857, 843, 1155, 4633, 662, 3531, 2082, 1077, 1630, 29951, 29889]



In [92]:

# Проверяем, что для токена "_за" один и тот же идентификатор
target_token = "▁за"

# Кодируем (токенизируем) текст
first_text_tokens = tokenizer.tokenize(texts[0])
second_text_tokens = tokenizer.tokenize(texts[1])

# получаем id токенов обоих текстов
first_text_token_ids = tokenizer.encode(texts[0],add_special_tokens=False)
second_text_token_ids = tokenizer.encode(texts[1],add_special_tokens=False)

# получаем индексы, на которых находится наш "замок"
first_position = first_text_tokens.index(target_token)
second_position = second_text_tokens.index(target_token)

print(f"ID токена '_за': \n- в первом тексте {first_text_token_ids[first_position]} \n- во втором тексте {second_text_token_ids[second_position]}")


ID токена '_за': 
- в первом тексте 1077 
- во втором тексте 1077


## Достаем таблицу входных эмбеддингов

In [ ]:
# получаем модель 'rubert-tiny2'

import torch
from transformers import AutoModel, AutoTokenizer

model_name = "cointegrated/rubert-tiny2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

model.eval()

config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

c:\Users\luxet\ai_assitant\aieng\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\luxet\.cache\huggingface\hub\models--cointegrated--rubert-tiny2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.74M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  118MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(83828, 312, padding_idx=0)
    (position_embeddings): Embedding(2048, 312)
    (token_type_embeddings): Embedding(2, 312)
    (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-2): 3 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=312, out_features=312, bias=True)
            (key): Linear(in_features=312, out_features=312, bias=True)
            (value): Linear(in_features=312, out_features=312, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=312, out_features=312, bias=True)
            (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=0

In [145]:
# найдем идентификатор слова 'замок' в текущем токенайзере

word = "замок"

tokens = tokenizer.tokenize(word)
token_ids = tokenizer.encode(word, add_special_tokens=False)

print(tokens, token_ids)

['замок'] [15006]


In [158]:
# Проверяем, что для токена "замок" один и тот же всегда идентификатор (в рамках текущего токенайзера)
target_token_zamok = "замок"

# Ттокенизируем текст
first_text_tokens = tokenizer.tokenize(texts[0])
second_text_tokens = tokenizer.tokenize(texts[1])

# получаем id токенов обоих текстов
first_text_token_ids = tokenizer.encode(texts[0],add_special_tokens=False)
second_text_token_ids = tokenizer.encode(texts[1],add_special_tokens=False)

# получаем индексы, на которых находится наш "замок"
first_position = first_text_tokens.index(target_token_zamok)
second_position = second_text_tokens.index(target_token_zamok)

print(f'Токены первого текста: {first_text_token_ids}, id токенов: {first_text_token_ids}')
print(f'Токены первого текста: {second_text_token_ids}, id токенов: {second_text_token_ids}')

print(f"\nID токена 'замок': \n- в первом тексте {first_text_token_ids[first_position]} \n- во втором тексте {second_text_token_ids[second_position]}")


Токены первого текста: [45069, 15006, 33766, 548, 58968, 18], id токенов: [45069, 15006, 33766, 548, 58968, 18]
Токены первого текста: [35773, 20132, 938, 3389, 16729, 15006, 18], id токенов: [35773, 20132, 938, 3389, 16729, 15006, 18]

ID токена 'замок': 
- в первом тексте 15006 
- во втором тексте 15006


In [155]:
# из модели достаем слой входных эмбеддингов
embedding_layer = model.get_input_embeddings() 

# выводим веса слоя
embedding_matrix = embedding_layer.weight

print('Размерность матрицы эмбеддингов: ', embedding_matrix.shape)
print('\nКонкретные числа зависят от выбранной модели, но смысл размерностей всегда один и тот же: \nпервое число — vocab_size (размер словаря токенизатора), \nвторое — d_model, длина вектора одного токена.') 

Размерность матрицы эмбеддингов:  torch.Size([83828, 312])

Конкретные числа зависят от выбранной модели, но смысл размерностей всегда один и тот же: 
первое число — vocab_size (размер словаря токенизатора), 
второе — d_model, длина вектора одного токена.


In [175]:
# Теперь достанем строку из матрицы эмбеддингов для 'замок'

token = tokens[0] # берем токен 'замок'
token_id = token_ids[0] # получаем id этого токена

token_embedding = embedding_matrix[token_id] # достаем строку из матрицы эмбеддингов для id токена

print(token)
print(token_id)
print(token_embedding[:10]) # выводим первые 10 чисел вектора токена 'замок'

замок
15006
tensor([ 0.0343, -0.1519, -0.0285,  0.0585,  0.0054, -0.0599, -0.0170, -0.1101,
        -0.0062,  0.0816], grad_fn=<SliceBackward0>)


## Normalization

In [ ]:
import torch.nn.functional as F

embedding_matrix = model.get_input_embeddings().weight.detach()

normalized_embeddings = F.normalize(embedding_matrix, p=2, dim=1) # отнормировали векторы в матрице эмбеддингов

print(normalized_embeddings.shape)

torch.Size([83828, 312])


In [182]:
# отнормированный вектор для 'замок' 

target_token = "замок"

target_ids = tokenizer.encode(target_token, add_special_tokens=False)

target_id = target_ids[0]
target_embedding = normalized_embeddings[target_id]

print(target_embedding.shape) 
print(target_embedding[:10])

torch.Size([312])
tensor([ 0.0294, -0.1300, -0.0244,  0.0501,  0.0046, -0.0513, -0.0145, -0.0942,
        -0.0053,  0.0699])


## Nearest tokens

In [185]:
# посчитаем косинусное сходство между выбранным токеном и всеми токенами из словаря
# т.к. мы отнормировали векторы, то скалярное произведение равнозначно косиносному сходству

similarities = normalized_embeddings @ target_embedding

print(similarities[:10]) # косинусное сходство от 0 до 1, чем ближе к 1, тем больше сходство
print(similarities.shape) # размерность равна количеству токенов в словаре

tensor([ 0.4613,  0.0682,  0.0979,  0.2047,  0.1637,  0.2804, -0.1648,  0.5024,
         0.4784,  0.3087])
torch.Size([83828])


In [186]:
# присваиваем сходству замка с замком значение = -1
similarities[target_id] = -1

# присваиваем всем специальным символам из словаря токенов косинусное сходство = -1
for special_token_id in tokenizer.all_special_ids:
    similarities[special_token_id] = -1

In [ ]:
# находим 10 ближайших токенов

top_k = 10

top_scores, top_ids = torch.topk(similarities, k=top_k) # получаем идентификаторы 10 самых похожих на 'замок' токенов

for score, token_id in zip(top_scores, top_ids):
    token = tokenizer.convert_ids_to_tokens(int(token_id))
    print(token, float(score)) 

замка 0.9055870771408081
крепость 0.8599364161491394
крепости 0.8574362397193909
дворец 0.8550406098365784
Замок 0.8355206251144409
монастырь 0.8268381357192993
король 0.8193625211715698
Храм 0.814708948135376
храм 0.8144423961639404
князь 0.7962237596511841


In [ ]:
# Функция, которая достает id токена

def get_single_token_id(token: str) -> int:
    token_ids = tokenizer.encode(token, add_special_tokens=False)
    token_parts = tokenizer.tokenize(token)

    if len(token_ids) != 1:
        raise ValueError(
            f"Слово {token!r} разбилось на несколько токенов: {token_parts}"
        )

    return token_ids[0]

get_single_token_id('улица')

9054

In [ ]:
# Функция, которая ищет ближайшие токены

def find_nearest_tokens(target_token: str, top_k: int = 10) -> list[tuple[str, float]]:
    # получаем id токена
    target_token_id = get_single_token_id(target_token)
    #target_token_id = tokenizer.encode(target_token, add_special_tokens=False)
    #target_token_id = target_token_id[0]

    # получаем матрицу эмбеддингов, нормализуем ее, достаем отнормализованный вектор для таргет токена
    embedding_matrix = model.get_input_embeddings().weight.detach()
    normalized_embeddings = F.normalize(embedding_matrix, p=2, dim=1)    
    target_embedding = normalized_embeddings[target_token_id]

    # находим косинусное сходство для каждого токена из словаря
    similarities = normalized_embeddings @ target_embedding


    # получаем id токенов, которые имеют самое большое сходство с таргет токеном
    top_scores, top_ids = torch.topk(similarities, k=top_k)

    # печатаем слова токенов
    for score, token_id in zip(top_scores, top_ids):
        token = tokenizer.convert_ids_to_tokens(int(token_id))
        print(token, float(score)) 


In [261]:
find_nearest_tokens('платье')

платье 1.0
платья 0.8789749145507812
белье 0.802545964717865
одеяло 0.7718174457550049
плащ 0.7617365121841431
печенье 0.7433220744132996
сиденье 0.7417525053024292
пиджа 0.7404322028160095
пальто 0.7354308366775513
джинсы 0.7335700988769531
